# Prepare and merge all data for Autoencoder

In [1]:
import pandas as pd
import re
from datetime import datetime

In [2]:
# Variables and Paths
ALL_DATA_CSV = "output/merged_data.csv"
LATENT_FILE = "output/Experiments/BetaScanVAE/latent_representations/train_mu_beta_3.00e-05.csv"
# LATENT_FILE = "output/Experiments/BetaScanVAE/latent_representations/val_mu_beta_3.00e-05.csv"
DICOM_FILE = "data/csvData/dicom_metadata.csv"
OUTPUT_FILE = "output/final_train_combined_vae_data.csv"
# OUTPUT_FILE = "output/final_validation_combined_vae_data.csv"

In [4]:
# Load merged clinical data
df_merged = pd.read_csv(ALL_DATA_CSV, low_memory=False)
print(f"Merged clinical data: {df_merged.shape}")

# Load latent vectors
df_latent = pd.read_csv(LATENT_FILE)
print(f"Latent vectors: {df_latent.shape}")

# Load DICOM metadata for scanner info
df_dicom = pd.read_csv(DICOM_FILE)
print(f"DICOM metadata: {df_dicom.shape}")

Merged clinical data: (41816, 42)
Latent vectors: (2380, 259)
DICOM metadata: (2986, 6)


In [6]:
df_latent.rename(columns={'file_path': 'FilePath'}, inplace=True)
df_latent_clean = df_latent.dropna(subset=['FilePath']).copy()
df_latent_clean.shape

(2380, 259)

In [7]:
df_latent_clean.head(2)

,FilePath,PATNO,label,latent_0,latent_1,latent_2,latent_3,latent_4,latent_5,latent_6,...,latent_246,latent_247,latent_248,latent_249,latent_250,latent_251,latent_252,latent_253,latent_254,latent_255
0,data/Images/PPMI_Images_PD/219605/Reconstructe...,219605,PD,-0.032396,0.101397,0.004257,-0.012963,0.086173,0.282817,0.379048,...,0.045734,-0.036642,0.395798,-0.092973,0.758001,-0.310369,0.080733,-0.050971,-0.105988,0.253290
1,data/Images/PPMI_Images_PD/3502/Reconstructed_...,3502,PD,0.191943,-0.082233,0.011299,-0.055740,-0.068799,-0.108955,0.467494,...,-0.079067,-0.202480,-0.368297,-0.147368,0.694748,-0.901797,0.181607,0.054750,0.169683,-0.081547


In [8]:
df_dicom.rename(columns={'file_path': 'FilePath'}, inplace=True)
df_dicom_latest = df_dicom.dropna(subset=['FilePath']).copy()
df_dicom_latest.shape

(2986, 6)

In [9]:
df_dicom_latest.head(2)

,FilePath,group,PatientSex,StudyDescription,Manufacturer,ManufacturerModelName
0,Images\PPMI_Images_PD\100001\Reconstructed_DaT...,PD,M,1-DAT,SIEMENS NM,Encore2
1,Images\PPMI_Images_PD\100001\Reconstructed_DaT...,PD,M,V02-DAT,SIEMENS NM,Encore2


In [10]:
print("--- Latent DataFrame Path Example ---")
print(df_latent_clean['FilePath'].iloc[0])

print("\n--- DICOM DataFrame Path Example ---")
print(df_dicom_latest['FilePath'].iloc[0])

--- Latent DataFrame Path Example ---
data/Images/PPMI_Images_PD/219605/Reconstructed_DaTSCAN/2023-04-06_14_52_54.0/I1698019/PPMI_219605_NM_Reconstructed_DaTSCAN_Br_20230508143813985_1_S1220401_I1698019.dcm

--- DICOM DataFrame Path Example ---
Images\PPMI_Images_PD\100001\Reconstructed_DaTSCAN\2020-09-09_17_07_33.0\I1452480\PPMI_100001_NM_Reconstructed_DaTSCAN_Br_20210608102518754_1_S1028880_I1452480.dcm


In [13]:
# Function to normalize paths
def normalize_path(path_str):
    if pd.isna(path_str): return path_str
    
    # 1. Force forward slashes
    clean_p = path_str.replace('\\', '/')
    
    # 2. Remove 'data/' prefix if it exists to ensure matching
    if clean_p.startswith('data/'):
        clean_p = clean_p.replace('data/', '')
        
    # 3. Strip any leading/trailing whitespace
    return clean_p.strip()

# Apply to BOTH dataframes
df_latent_clean['Merge_Key'] = df_latent_clean['FilePath'].apply(normalize_path)
df_dicom_latest['Merge_Key'] = df_dicom_latest['FilePath'].apply(normalize_path)

# Check if they look the same now
print("New Key Latent:", df_latent_clean['Merge_Key'].iloc[0])
print("New Key DICOM: ", df_dicom_latest['Merge_Key'].iloc[0])

New Key Latent: Images/PPMI_Images_PD/219605/Reconstructed_DaTSCAN/2023-04-06_14_52_54.0/I1698019/PPMI_219605_NM_Reconstructed_DaTSCAN_Br_20230508143813985_1_S1220401_I1698019.dcm
New Key DICOM:  Images/PPMI_Images_PD/100001/Reconstructed_DaTSCAN/2020-09-09_17_07_33.0/I1452480/PPMI_100001_NM_Reconstructed_DaTSCAN_Br_20210608102518754_1_S1028880_I1452480.dcm


In [15]:
df_latent_with_scanner = pd.merge(
    df_latent_clean,
    df_dicom_latest[['Merge_Key', 'Manufacturer', 'ManufacturerModelName']],
    on='Merge_Key',
    how='left'  # Keep all latent vectors, add scanner info
)
df_latent_with_scanner.shape

(2380, 262)

In [17]:
df_latent_with_scanner['FilePath'] = df_latent_with_scanner['Merge_Key']

In [18]:
df_latent_with_scanner.sample(5)

,FilePath,PATNO,label,latent_0,latent_1,latent_2,latent_3,latent_4,latent_5,latent_6,...,latent_249,latent_250,latent_251,latent_252,latent_253,latent_254,latent_255,Merge_Key,Manufacturer,ManufacturerModelName
835,Images/PPMI_Images_PD/4073/Reconstructed_DaTSC...,4073,PD,0.199038,-0.115835,-0.048101,0.062181,-0.061017,0.120468,0.408389,...,-0.108601,1.976005,-0.089456,-0.057015,0.125152,0.215394,0.106095,Images/PPMI_Images_PD/4073/Reconstructed_DaTSC...,SIEMENS NM,Encore2
2093,Images/PPMI_Images_PD/40714/Reconstructed_DaTS...,40714,PD,0.309466,-0.104393,0.118039,0.062527,0.067206,0.293246,-0.381693,...,0.047651,-0.085594,-0.028028,0.080627,0.033196,0.177785,0.122963,Images/PPMI_Images_PD/40714/Reconstructed_DaTS...,GE MEDICAL SYSTEMS,INFINIA
625,Images/PPMI_Images_PD/3589/Reconstructed_DaTSC...,3589,PD,-0.157191,0.047951,0.022061,0.194309,0.185338,0.206991,-0.488152,...,0.021606,-0.599611,-0.568788,-0.172339,-0.060166,-0.028564,0.252535,Images/PPMI_Images_PD/3589/Reconstructed_DaTSC...,SIEMENS NM,Encore2
2217,Images/PPMI_Images_PD/3826/Reconstructed_DaTSC...,3826,PD,-0.100947,-0.021555,0.080365,-0.054547,-0.058971,0.131521,0.005411,...,0.005821,1.073262,-0.109225,-0.031848,-0.001154,-0.060652,0.026049,Images/PPMI_Images_PD/3826/Reconstructed_DaTSC...,SIEMENS NM,Encore2
1431,Images/PPMI_Images_PD/114265/Reconstructed_DaT...,114265,PD,0.210976,0.060132,0.053869,0.051226,0.017899,0.031964,1.074577,...,-0.027785,0.106555,0.562519,-0.181100,0.078798,0.125469,0.001964,Images/PPMI_Images_PD/114265/Reconstructed_DaT...,SIEMENS NM,Encore2


In [33]:
# 1. Robust Date Extraction (Finds YYYY-MM-DD anywhere in path)
def get_date_from_path(path_str):
    if pd.isna(path_str):
        return None
    
    # Regex to find pattern: 4 digits - 2 digits - 2 digits
    match = re.search(r'(\d{4}-\d{2}-\d{2})', str(path_str))
    if match:
        date_raw = match.group(1) # Extracts '2021-04-06'
        try:
            # Added datetime import requirement and better error handling
            return datetime.strptime(date_raw, '%Y-%m-%d').strftime('%m/%Y')
        except Exception:
            return None
    return None

In [34]:
# 2. Apply the fix
df_latent_with_scanner['DATSCAN_DATE'] = df_latent_with_scanner['FilePath'].apply(get_date_from_path)

# Verify we actually have dates now (Safe check)
dates_found = df_latent_with_scanner['DATSCAN_DATE'].dropna()
if not dates_found.empty:
    print("Latent Date Sample:", dates_found.iloc[0])
else:
    print("Warning: No dates could be extracted from FilePath. Check your regex or path format.")

df_latent_with_scanner.sample(2)

Latent Date Sample: 04/2023


,FilePath,PATNO,label,latent_0,latent_1,latent_2,latent_3,latent_4,latent_5,latent_6,...,latent_250,latent_251,latent_252,latent_253,latent_254,latent_255,Merge_Key,Manufacturer,ManufacturerModelName,DATSCAN_DATE
1079,Images/PPMI_Images_Cont/3570/Reconstructed_DaT...,3570,Control,0.187724,-0.014511,0.164907,0.134069,-0.018410,-0.069411,0.558088,...,1.143505,0.231231,0.195241,0.043377,-0.000560,-0.002976,Images/PPMI_Images_Cont/3570/Reconstructed_DaT...,SIEMENS NM,Encore2,09/2011
1611,Images/PPMI_Images_Cont/3917/Reconstructed_DaT...,3917,Control,0.025759,0.127676,0.185345,-0.134088,0.039579,0.132617,0.075522,...,1.513131,-0.158246,-0.209293,-0.179626,-0.065101,0.263881,Images/PPMI_Images_Cont/3917/Reconstructed_DaT...,SIEMENS NM,ENCORE2,04/2013


In [35]:

# 3. Clean Clinical Data (df_merged)
df_merged['PATNO'] = pd.to_numeric(df_merged['PATNO'], errors='coerce').fillna(0).astype(int)
df_latent_with_scanner['PATNO'] = pd.to_numeric(df_latent_with_scanner['PATNO'], errors='coerce').fillna(0).astype(int)

In [36]:
df_latent_with_scanner.sample(2)

,FilePath,PATNO,label,latent_0,latent_1,latent_2,latent_3,latent_4,latent_5,latent_6,...,latent_250,latent_251,latent_252,latent_253,latent_254,latent_255,Merge_Key,Manufacturer,ManufacturerModelName,DATSCAN_DATE
1271,Images/PPMI_Images_PD/142629/Reconstructed_DaT...,142629,PD,-0.053215,0.122663,-0.029001,-0.046328,0.023719,0.149388,0.450400,...,-0.535631,0.156641,0.160722,-0.065495,-0.137380,0.110505,Images/PPMI_Images_PD/142629/Reconstructed_DaT...,GE MEDICAL SYSTEMS,Tandem_Optima_640,02/2022
1824,Images/PPMI_Images_PD/3021/Reconstructed_DaTSC...,3021,PD,-0.019085,0.077575,0.104339,-0.157977,0.061297,0.191006,-0.177517,...,0.410872,0.205410,-0.124951,-0.088508,0.241868,0.090218,Images/PPMI_Images_PD/3021/Reconstructed_DaTSC...,Philips Healthcare,BrightView,06/2013


In [40]:
df_latent_with_scanner.shape

(2380, 263)

In [38]:
# Convert clinical dates to strings, handle NaNs
df_merged['DATSCAN_DATE'] = pd.to_datetime(
    df_merged['DATSCAN_DATE'], 
    format='mixed', 
    errors='coerce'
).dt.strftime('%m/%Y')

df_merged.sample(2)

,PATNO,EVENT_ID,AGE_AT_VISIT,BIRTHDT,SEX,INFODT,REC_ID,PAG_NAME,AFICBERB,ASHKJEW,...,DATSCAN_DATE,DATSCAN_CAUDATE_R,DATSCAN_CAUDATE_L,DATSCAN_PUTAMEN_R,DATSCAN_PUTAMEN_L,DATSCAN_PUTAMEN_R_ANT,DATSCAN_PUTAMEN_L_ANT,DATSCAN_ANALYZED,DATSCAN_NOT_ANALYZED_REASON,DATSCAN_OTHER_SPECIFY
23197,75421,R12,82.7,07/1942,1.0,10/2018,IA87892,SCREEN,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22693,73467,V02,67.3,02/1952,1.0,11/2018,676302501,SCREEN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [41]:
df_merged.shape

(41816, 42)

In [39]:
# 4. Perform the Merge
df_combined = pd.merge(
    df_latent_with_scanner, 
    df_merged,
    on=['PATNO', 'DATSCAN_DATE'],
    how='inner'
)

print(f"Merge Shape: {df_combined.shape}")

Merge Shape: (2373, 303)


In [43]:
df_combined['DATSCAN_DATE'].sample(5)

1912    02/2023
1907    08/2017
2225    04/2014
99      03/2021
1398    07/2022
Name: DATSCAN_DATE, dtype: object

In [44]:
df_combined.sample(5)

,FilePath,PATNO,label,latent_0,latent_1,latent_2,latent_3,latent_4,latent_5,latent_6,...,DATSCAN_LIGAND,DATSCAN_CAUDATE_R,DATSCAN_CAUDATE_L,DATSCAN_PUTAMEN_R,DATSCAN_PUTAMEN_L,DATSCAN_PUTAMEN_R_ANT,DATSCAN_PUTAMEN_L_ANT,DATSCAN_ANALYZED,DATSCAN_NOT_ANALYZED_REASON,DATSCAN_OTHER_SPECIFY
2071,Images/PPMI_Images_PD/40671/Reconstructed_DaTS...,40671,PD,0.035657,0.058342,-0.013710,0.028277,0.086209,0.234628,-0.078048,...,NaN,1.84,1.80,0.50,0.82,0.95,1.20,Yes,NaN,NaN
184,Images/PPMI_Images_PD/3439/Reconstructed_DaTSC...,3439,PD,-0.251017,0.090016,0.085982,0.121626,0.137607,0.295092,0.264122,...,NaN,2.87,2.79,1.07,0.75,1.74,1.60,Yes,NaN,NaN
1529,Images/PPMI_Images_PD/3752/Reconstructed_DaTSC...,3752,PD,-0.024303,0.010473,0.047183,-0.036000,0.094356,0.044024,-0.249694,...,NaN,1.36,1.73,0.30,0.83,0.76,1.20,Yes,NaN,NaN
1110,Images/PPMI_Images_Cont/3424/Reconstructed_DaT...,3424,Control,-0.054375,0.121774,0.215598,-0.197770,-0.118930,-0.029603,-0.046172,...,NaN,4.96,4.61,3.76,3.63,4.22,4.36,Yes,NaN,NaN
1802,Images/PPMI_Images_PD/53600/Reconstructed_DaTS...,53600,PD,-0.073725,0.047275,0.010724,0.200625,0.137479,-0.229608,-0.349729,...,NaN,1.04,1.51,0.39,0.76,0.63,1.04,Yes,NaN,NaN


In [46]:
"Manufacturer" in df_combined.columns

True

In [47]:
# Save the final merged dataset for the Autoencoder

df_combined.to_csv(OUTPUT_FILE, index=False)

print(f"Successfully saved merged data to: {OUTPUT_FILE}")
print(f"Final file contains {df_combined.shape[0]} rows and {df_combined.shape[1]} columns.")

Successfully saved merged data to: output/final_train_combined_vae_data.csv
Final file contains 2373 rows and 303 columns.
